# Generalized Linear Models — one framework, many regressions

> Tutorial pair for [`generalized_linear_models.py`](generalized_linear_models.py).

## 1. Intuition
Ordinary least squares quietly assumes the response is Gaussian with constant
variance. That is wrong for **counts** (non-negative integers, variance grows
with the mean) and for **positive skewed amounts** (waiting times, insurance
costs). GLMs keep the familiar linear predictor $\eta = Xw$ but feed it through a
**link function** so the model's mean lives in the right space, and they model
the noise with the matching **exponential-family** distribution. Pick the
family + link and you recover OLS, logistic regression, **Poisson** regression
(counts), **Gamma** regression (positive skew), and more — all fit by the *same*
algorithm.

## 2. Concept (the slide)
- **Random component:** $y_i$ drawn from an exponential-family distribution with
  mean $\mu_i$ (Poisson, Gamma, Gaussian, Bernoulli, ...).
- **Systematic component:** a linear predictor $\eta_i = x_i^\top w$.
- **Link:** an invertible $g$ with $g(\mu_i) = \eta_i$, so $\mu_i = g^{-1}(\eta_i)$.
  We use the **log link** $g(\mu)=\log\mu \Rightarrow \mu=e^{\eta}$, which keeps
  $\mu>0$ — exactly what counts and amounts need.
- **Fit:** maximize the log-likelihood. Two routes: **IRLS / Fisher scoring**
  (Newton's method, a few iterations) and **gradient descent** on the NLL.

## 3. Math derivation — exponential family, score, and IRLS

### Exponential family
Write the density in canonical form
$$p(y\mid\theta,\phi)=\exp\!\Big(\frac{y\theta-b(\theta)}{a(\phi)}+c(y,\phi)\Big).$$
Standard identities give the mean and variance from $b$:
$$\mu=\mathbb E[y]=b'(\theta),\qquad \operatorname{Var}(y)=a(\phi)\,b''(\theta)=a(\phi)\,V(\mu),$$
where $V(\mu)=b''(\theta)$ is the **variance function**. The model couples $\mu$
to the linear predictor via the link $g(\mu)=\eta=x^\top w$.

### Score (gradient of the log-likelihood)
For one observation, by the chain rule
$$\frac{\partial \ell}{\partial w}
 =\underbrace{\frac{y-\mu}{a(\phi)\,V(\mu)}}_{\partial\ell/\partial\mu\;\cdot\;?}\,
   \frac{d\mu}{d\eta}\,x .$$
Stacking all $n$ rows, the **score** is
$$\nabla_w \ell = X^\top D\,(y-\mu),\qquad
  D=\operatorname{diag}\!\Big(\tfrac{1}{V(\mu_i)}\tfrac{d\mu_i}{d\eta_i}\Big)
  \;(\text{absorb }a(\phi)).$$

### Two families with the log link ($\mu=e^\eta\Rightarrow d\mu/d\eta=\mu$)

**Poisson.** $V(\mu)=\mu$, so $\tfrac{1}{V}\tfrac{d\mu}{d\eta}=\tfrac{1}{\mu}\mu=1$:
$$\ell=\sum_i\big(y_i\eta_i-e^{\eta_i}\big)+\text{const},\qquad
  \nabla_w(-\ell)=-X^\top(y-\mu).$$

**Gamma (unit shape).** $V(\mu)=\mu^2$, so $\tfrac{1}{V}\tfrac{d\mu}{d\eta}=\tfrac{1}{\mu^2}\mu=\tfrac1\mu$:
$$-\ell\;\propto\;\sum_i\Big(\frac{y_i}{\mu_i}+\eta_i\Big),\qquad
  \nabla_w(-\ell)=-X^\top\frac{y-\mu}{\mu}.$$

### IRLS = Fisher scoring = Newton with the expected Hessian
Newton's update is $w\leftarrow w-H^{-1}\nabla(-\ell)$. Replacing the Hessian by
its expectation (the **Fisher information** $\mathcal I = X^\top W X$ with working
weights $W_i=\big(\tfrac{d\mu_i}{d\eta_i}\big)^2/V(\mu_i)$) and rearranging turns
each step into a **weighted least squares** on a *working response*:
$$z_i=\eta_i+(y_i-\mu_i)\frac{d\eta_i}{d\mu_i},\qquad
  \boxed{\,w^{+}=\big(X^\top W X\big)^{-1}X^\top W z\,}.$$
For the log link, $d\eta/d\mu=1/\mu$, so $z=\eta+(y-\mu)/\mu$; the weights are
$W=\mu$ (Poisson) and $W=\mathbf 1$ (Gamma). Each iteration solves one weighted
normal-equations system — Newton-fast, typically a handful of steps.

## 4. NumPy implementation — IRLS *and* gradient descent, by hand

In [ ]:
# ===== actual implementation from generalized_linear_models.py =====
from __future__ import annotations

import numpy as np

SEED = 0

import torch

import torch.nn as nn

def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

def _make_poisson(rng, n=400, d=3):
    """Synthetic counts: y ~ Poisson(exp(Xw + b))."""
    X = rng.normal(size=(n, d)) * 0.5
    w_true = np.array([0.6, -0.4, 0.3])
    eta = 0.5 + X @ w_true
    y = rng.poisson(np.exp(eta)).astype(float)
    return X, y, np.r_[0.5, w_true]

def _make_gamma(rng, n=400, d=3, shape=5.0):
    """Synthetic positive responses: y ~ Gamma(mean=exp(Xw+b))."""
    X = rng.normal(size=(n, d)) * 0.5
    w_true = np.array([0.5, 0.2, -0.3])
    mu = np.exp(0.2 + X @ w_true)
    # numpy gamma is parameterized by (shape k, scale theta), mean = k*theta
    y = rng.gamma(shape=shape, scale=mu / shape)
    return X, y, np.r_[0.2, w_true]

def demo():
    np.random.seed(SEED); torch.manual_seed(SEED)
    rng = np.random.default_rng(SEED)

    print("=== Poisson regression (log link) ===")
    Xp, yp, w_true_p = _make_poisson(rng)
    irls = GLMNumPy(family="poisson", fit_method="irls", n_iters=50).fit(Xp, yp)
    gd = GLMNumPy(family="poisson", fit_method="gd", lr=0.2, n_iters=400).fit(Xp, yp)
    tt = GLMTorch(Xp.shape[1], family="poisson").fit(Xp, yp, lr=0.05, n_iters=600)
    print(f"  true coefs (b,w):  {np.round(w_true_p, 3)}")
    print(f"  IRLS coefs:        {np.round(irls.w, 3)}  (final NLL {irls.history[-1]:.4f})")
    print(f"  GD   coefs:        {np.round(gd.w, 3)}  (final NLL {gd.history[-1]:.4f})")
    tw = np.r_[tt.linear.bias.item(), tt.linear.weight.detach().cpu().numpy().ravel()]
    print(f"  Torch coefs:       {np.round(tw, 3)}")
    print(f"  IRLS converged in {len(irls.history)} iters (Newton is fast)")

    print("\n=== Gamma regression (log link) ===")
    Xg, yg, w_true_g = _make_gamma(rng)
    irls_g = GLMNumPy(family="gamma", fit_method="irls", n_iters=50).fit(Xg, yg)
    gd_g = GLMNumPy(family="gamma", fit_method="gd", lr=0.2, n_iters=400).fit(Xg, yg)
    tt_g = GLMTorch(Xg.shape[1], family="gamma").fit(Xg, yg, lr=0.05, n_iters=600)
    print(f"  true coefs (b,w):  {np.round(w_true_g, 3)}")
    print(f"  IRLS coefs:        {np.round(irls_g.w, 3)}")
    print(f"  GD   coefs:        {np.round(gd_g.w, 3)}")
    twg = np.r_[tt_g.linear.bias.item(), tt_g.linear.weight.detach().cpu().numpy().ravel()]
    print(f"  Torch coefs:       {np.round(twg, 3)}")

    # quality: mean absolute relative error on the mean
    mu_hat = irls.predict(Xp)
    print(f"\n  Poisson IRLS mean |y-mu|: {np.mean(np.abs(yp - mu_hat)):.3f}")


class GLMNumPy:
    r"""
    Generalized Linear Model with a log link, fit by IRLS or gradient descent.

    Linear predictor:  eta = X w           (X includes a bias column of ones)
    Mean:              mu  = g^{-1}(eta) = exp(eta)         (log link)

    POISSON  y ~ Poisson(mu):
        log-lik (drop const):  L = sum_i [ y_i eta_i - exp(eta_i) ]
        gradient (of -L):      -X^T (y - mu)
        IRLS weights:          W = mu      (variance fn V(mu)=mu, log link)

    GAMMA  y ~ Gamma(mean=mu, shape=nu), log link:
        log-lik (drop const):  L = sum_i [ -y_i/mu_i - eta_i ]  (times nu)
        gradient (of -L):      -X^T (y - mu)/mu
        IRLS weights:          W = 1       (variance fn V(mu)=mu^2, log link)
    """

    def __init__(self, family="poisson", fit_method="irls", lr=0.01,
                 n_iters=100, l2=0.0, seed=SEED):
        assert family in ("poisson", "gamma")
        assert fit_method in ("irls", "gd")
        self.family = family
        self.fit_method = fit_method
        self.lr = lr
        self.n_iters = n_iters
        self.l2 = l2                       # ridge penalty on weights (not bias)
        self.seed = seed
        self.w = None
        self.history = []                  # negative log-likelihood per iter

    # --- design matrix: prepend a column of ones for the intercept ----------
    @staticmethod
    def _design(X):
        X = np.asarray(X, float)
        return np.hstack([np.ones((len(X), 1)), X])

    # --- inverse link: log link => mu = exp(eta) ----------------------------
    def _inv_link(self, eta):
        return np.exp(np.clip(eta, -30, 30))

    # --- per-family negative log-likelihood (for monitoring) ----------------
    def _nll(self, eta, y):
        mu = self._inv_link(eta)
        if self.family == "poisson":
            # -[y*eta - mu]  (drop log(y!) constant, independent of w)
            return np.mean(mu - y * eta)
        else:  # gamma, unit shape -> deviance-like NLL: y/mu + eta
            return np.mean(y / mu + eta)

    # --- gradient of the NEGATIVE log-likelihood wrt w ----------------------
    def _grad(self, Xb, eta, y):
        mu = self._inv_link(eta)
        if self.family == "poisson":
            # dL/dw = X^T (y - mu);  grad of -L = -X^T (y - mu)
            g = -Xb.T @ (y - mu) / len(y)
        else:  # gamma with log link: dL/dw = X^T (y - mu)/mu
            g = -Xb.T @ ((y - mu) / mu) / len(y)
        if self.l2:
            reg = self.l2 * self.w
            reg[0] = 0.0                   # don't penalize the intercept
            g = g + reg
        return g

    # --- IRLS: Fisher scoring solves a weighted least squares each step -----
    def _irls(self, Xb, y):
        n, d = Xb.shape
        w = np.zeros(d)
        # warm start the intercept at log(mean(y)) so exp(eta0)=mean(y)
        w[0] = np.log(max(y.mean(), 1e-3))
        for _ in range(self.n_iters):
            eta = Xb @ w
            mu = self._inv_link(eta)
            # Working weights W and working response z (Fisher scoring):
            #   z = eta + (y - mu) * (deta/dmu)
            # log link => deta/dmu = 1/mu.
            if self.family == "poisson":
                W = mu                       # V(mu)=mu
                z = eta + (y - mu) / mu
            else:  # gamma: V(mu)=mu^2, W = (dmu/deta)^2 / V = mu^2/mu^2 = 1
                W = np.ones_like(mu)
                z = eta + (y - mu) / mu
            # Solve weighted normal equations: (X^T W X + l2 I) w = X^T W z
            WX = Xb * W[:, None]
            A = Xb.T @ WX
            if self.l2:
                R = self.l2 * n * np.eye(d); R[0, 0] = 0.0
                A = A + R
            b = Xb.T @ (W * z)
            w_new = np.linalg.solve(A, b)
            self.history.append(self._nll(Xb @ w_new, y))
            if np.linalg.norm(w_new - w) < 1e-8:
                w = w_new
                break
            w = w_new
        return w

    # --- plain gradient descent on the NLL ----------------------------------
    def _gd(self, Xb, y):
        rng = np.random.default_rng(self.seed)
        w = np.zeros(Xb.shape[1])
        w[0] = np.log(max(y.mean(), 1e-3))
        self.w = w
        for _ in range(self.n_iters):
            eta = Xb @ w
            self.history.append(self._nll(eta, y))
            g = self._grad(Xb, eta, y)
            w = w - self.lr * g
            self.w = w
        return w

    def fit(self, X, y):
        y = np.asarray(y, float)
        Xb = self._design(X)
        self.w = np.zeros(Xb.shape[1])
        self.history = []
        self.w = self._irls(Xb, y) if self.fit_method == "irls" else self._gd(Xb, y)
        return self

    def predict(self, X):
        """Return the predicted mean response mu = exp(Xw)."""
        return self._inv_link(self._design(X) @ self.w)

## 5. PyTorch implementation — minimize the same NLL with autograd

In [ ]:
# ===== actual implementation from generalized_linear_models.py =====
class GLMTorch(nn.Module):
    """
    GLM (log link) trained by minimizing the negative log-likelihood with autograd.

    The forward pass produces the linear predictor eta = Xw + b; the loss is the
    per-family NLL. Autograd then reproduces exactly the hand-derived gradients.
    """

    def __init__(self, n_features, family="poisson"):
        super().__init__()
        assert family in ("poisson", "gamma")
        self.family = family
        self.linear = nn.Linear(n_features, 1)

    def forward(self, X):
        return self.linear(X).squeeze(-1)          # eta

    def nll(self, eta, y):
        mu = torch.exp(torch.clamp(eta, -30, 30))
        if self.family == "poisson":
            return (mu - y * eta).mean()           # drop log(y!) const
        return (y / mu + eta).mean()               # gamma, unit shape

    def fit(self, X, y, lr=0.05, n_iters=300, l2=0.0):
        dev = get_device(); self.to(dev)
        X = torch.as_tensor(X, dtype=torch.float32, device=dev)
        y = torch.as_tensor(y, dtype=torch.float32, device=dev)
        # warm start intercept at log(mean(y))
        with torch.no_grad():
            self.linear.bias.fill_(float(np.log(max(y.mean().item(), 1e-3))))
        opt = torch.optim.Adam(self.parameters(), lr=lr, weight_decay=l2)
        self.history = []
        for _ in range(n_iters):
            opt.zero_grad()
            loss = self.nll(self(X), y)
            loss.backward(); opt.step()
            self.history.append(loss.item())
        return self

    @torch.no_grad()
    def predict(self, X):
        dev = next(self.parameters()).device
        X = torch.as_tensor(X, dtype=torch.float32, device=dev)
        return torch.exp(self(X)).cpu().numpy()

## 6. Train — IRLS vs GD vs Torch on synthetic Poisson and Gamma data

In [ ]:
demo()

## 7. Visualization — IRLS converges in a few Newton steps; GD crawls

In [ ]:
import matplotlib; matplotlib.use("Agg")
import numpy as np, matplotlib.pyplot as plt
import generalized_linear_models as M

rng = np.random.default_rng(0)
Xp, yp, _ = M._make_poisson(rng)

irls = M.GLMNumPy(family="poisson", fit_method="irls", n_iters=50).fit(Xp, yp)
gd   = M.GLMNumPy(family="poisson", fit_method="gd", lr=0.2, n_iters=400).fit(Xp, yp)

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(irls.history, "o-", label="IRLS (Newton)")
ax[0].plot(gd.history, "-", label="gradient descent")
ax[0].set_xlabel("iteration"); ax[0].set_ylabel("negative log-likelihood")
ax[0].set_title("Poisson NLL vs iteration"); ax[0].legend(); ax[0].grid(True, alpha=.3)

mu_hat = irls.predict(Xp)
order = np.argsort(mu_hat)
ax[1].scatter(mu_hat, yp, s=10, alpha=.4, label="observed count")
ax[1].plot(mu_hat[order], mu_hat[order], "r-", label="predicted mean $\\mu$")
ax[1].set_xlabel("predicted mean $\\mu=e^{Xw}$"); ax[1].set_ylabel("y")
ax[1].set_title("Poisson fit: y vs predicted mean"); ax[1].legend()
plt.tight_layout(); plt.show()

## 8. Takeaways & pitfalls
- **Pick the family for the data**: counts → Poisson; positive skewed amounts →
  Gamma; binary → Bernoulli (logistic); real-valued → Gaussian (OLS). All are the
  *same* GLM machinery with a different variance function and link.
- The **log link** guarantees $\mu>0$ and makes coefficients multiplicative:
  $e^{w_j}$ is the factor by which the mean changes per unit of $x_j$.
- **IRLS is Fisher scoring**: each step is one weighted least squares; it usually
  converges in a few iterations, while plain GD needs hundreds and a tuned LR —
  but GD scales to huge data / streaming where forming $X^\top W X$ is costly.
- Poisson regression assumes mean = variance; real counts are often
  **overdispersed** (variance > mean) — then use Negative Binomial or a
  quasi-Poisson dispersion estimate.